In [19]:
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET
import json
import base64
import shutil
import tempfile
from pandas import DataFrame
from pathlib import Path

# Extract Data Range

In [ ]:
# --- Configuration ---
input_file = (
    "C:\\Users\\foolu\\AppData\\Roaming\\Microsoft\\Excel\\XL-Int (version 1).xlsb"
)
sheet_name = "XL-Int (version 2) csv.xlsb"
output_file = "C:\\Users\\foolu\\Workspace\\Excel\\BigInt\\extracted_data1.csv"

# Define your range
# 'usecols' defines the columns (e.g., "A:E")
# 'skiprows' defines where the range starts (0-indexed)
# 'nrows' defines how many rows to pull
target_cols = "A:D"
blank_rows = 1
total_rows = 1990

try:
	# 1. Extract the specific range
	df: DataFrame = pd.read_excel(
		input_file,
		sheet_name=sheet_name,
		usecols=target_cols,
		engine="pyxlsb",
		skiprows=blank_rows,
		nrows=total_rows,
	)

	# 2. Rewrite to CSV
	# index=False prevents pandas from adding an extra column for row numbers
	df.to_csv(output_file, index=False)

	print(f"Successfully extracted range to {output_file}")

except Exception as e:
	print(f"An error occurred: {e}")

Successfully extracted range to C:\Users\foolu\Workspace\Excel\BigInt\extracted_data1.csv


# Extract Function Modules

In [22]:
excel_file = Path('XL-Int.xlsx')
output_dir = Path('src')

# 1. Ensure the output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Temporary directory guarantees cleanup
with tempfile.TemporaryDirectory() as temp_dir:
    temp_excel_path = Path(temp_dir) / 'temp_copy.xlsx'

    try: # Bypass Windows locks if Excel is open
        shutil.copy2(excel_file, temp_excel_path)
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{excel_file}'.")
        exit(1)

    # 3. Open the temporary, unlocked copy
    with zipfile.ZipFile(temp_excel_path, 'r') as z:
        try:
            xml_content = z.read('customXml/item1.xml')
        except KeyError:
            print(f"❌ Error: No Advanced Formula Editor data found in '{excel_file}'.")
            exit(1)

        root = ET.fromstring(xml_content)

        # Extract and decode the Base64 UTF-16 string
        b64_text = "".join(root.itertext()).strip()
        raw_json_string = base64.b64decode(b64_text).decode('utf-16-le')

        afe_workspace = json.loads(raw_json_string)
        files = afe_workspace.get('files', [])

        extracted_count = 0

        # 4. Extract and save
        for file_obj in files:
            path = file_obj.get('path', '')
            module_code = file_obj.get('text', '')
            module_name = path.split('/')[-1]

            if not module_name or not module_code.strip():
                continue

            print(f"Extracting: {module_name}")

            output_file = output_dir / f"{module_name}.txt"
            output_file.write_text(module_code, encoding="utf-8")

            extracted_count += 1

print(f"\n✅ Successfully extracted {extracted_count} modules to the '{output_dir}' directory.")

Extracting: test_BigInt
Extracting: core_BigInt
Extracting: BigInt

✅ Successfully extracted 3 modules to the 'src' directory.
